# Week 5 Exercise – Simple RAG Q&A (GOlivierNation)

Minimal **Retrieval-Augmented Generation** demo: embed a small knowledge base, retrieve relevant chunks, and answer questions with the LLM. Uses OpenAI (or OpenRouter) for embeddings and chat; in-memory similarity search so no extra vector DB is required.

**Scope:** Only `week5/community-contributions/GOlivierNation/`. No other weeks touched.

In [ ]:
# Imports
import os
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Config: OpenRouter or OpenAI
load_dotenv(override=True)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENROUTER_BASE = "https://openrouter.ai/api/v1"
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL_OR = "openai/gpt-4o-mini"
CHAT_MODEL_OA = "gpt-4o-mini"

if OPENROUTER_API_KEY:
    client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE)
    chat_model = CHAT_MODEL_OR
    embed_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else OpenAI()
    print("Using OpenRouter for chat; OpenAI for embeddings (set OPENAI_API_KEY).")
elif OPENAI_API_KEY:
    client = OpenAI(api_key=OPENAI_API_KEY)
    embed_client = client
    chat_model = CHAT_MODEL_OA
    print("Using OpenAI.")
else:
    client = OpenAI()
    embed_client = client
    chat_model = CHAT_MODEL_OA
    print("Using default client; set OPENROUTER_API_KEY or OPENAI_API_KEY in .env.")

In [ ]:
# Small in-notebook knowledge base (no external files)
KNOWLEDGE = [
    "Insurellm is an insurance technology company. It offers products for health, auto, and property insurance.",
    "RAG stands for Retrieval-Augmented Generation. You retrieve relevant documents and then generate an answer using an LLM.",
    "Embeddings are vector representations of text. Similar texts have similar embeddings.",
    "Chunking splits long documents into smaller pieces so they fit context and improve retrieval.",
]

def get_embedding(text: str):
    r = embed_client.embeddings.create(input=[text], model=EMBED_MODEL)
    return np.array(r.data[0].embedding, dtype=np.float32)

def build_index():
    vectors = np.stack([get_embedding(s) for s in KNOWLEDGE])
    return vectors, KNOWLEDGE

VECTORS, TEXTS = build_index()

In [ ]:
def retrieve(query: str, top_k: int = 2):
    q = get_embedding(query)
    scores = np.dot(VECTORS, q) / (np.linalg.norm(VECTORS, axis=1) * np.linalg.norm(q) + 1e-9)
    idx = np.argsort(scores)[::-1][:top_k]
    return [TEXTS[i] for i in idx]

def rag_answer(question: str) -> str:
    if not (question or question.strip()):
        return ""
    chunks = retrieve(question.strip())
    context = "\n".join(chunks)
    try:
        r = client.chat.completions.create(
            model=chat_model,
            messages=[
                {"role": "system", "content": "Answer only from the given context. If the context does not contain the answer, say so."},
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
            ],
        )
        return (r.choices[0].message.content or "").strip()
    except Exception as e:
        return f"Error: {e}"

In [ ]:
# Gradio UI (Week 5 only)
with gr.Blocks(title="Week 5 – RAG Q&A", theme=gr.themes.Soft()) as demo:
    gr.Markdown("## Week 5 Exercise: Simple RAG Q&A")
    q = gr.Textbox(label="Question", placeholder="Ask about Insurellm, RAG, embeddings, or chunking...")
    out = gr.Textbox(label="Answer", lines=6)
    btn = gr.Button("Ask")
    btn.click(fn=rag_answer, inputs=q, outputs=out)
    q.submit(fn=rag_answer, inputs=q, outputs=out)
demo.launch(inbrowser=True)